# FreshPrice AI — Colab Training Notebook

**OpenEnv Hackathon | Meta × HuggingFace × PyTorch**

This notebook trains Qwen-2.5-7B on the FreshPrice perishable goods RL environment.
It uses **Unsloth** for 4-bit QLoRA training and **HF TRL** for GRPO.

### Minimum Requirements Covered
- Usage of OpenEnv (latest release)
- Training script using Unsloth + HF TRL
- Environment hosted on HuggingFace Spaces

### Runtime Requirements
- GPU: A100 (recommended) or T4 (slower)
- RAM: 16GB+
- Disk: 30GB+ for model weights

**Estimated time:** ~2 hours for SFT + GRPO Level 0

## 0. GPU Check

In [ ]:
!nvidia-smi
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

## 1. Install Dependencies

In [ ]:
# Unsloth + dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps trl peft accelerate bitsandbytes -q

# OpenEnv + project deps
!pip install openenv-core>=0.2.0 gymnasium wandb -q

In [ ]:
# Clone the repo (or mount from Drive)
import os
if not os.path.exists('QStorePrice'):
    !git clone https://github.com/YOUR_USERNAME/QStorePrice.git  # replace with your repo
%cd QStorePrice

# Install remaining requirements
!pip install -r requirements_training.txt -q

## 2. Verify Environment

In [ ]:
import sys
sys.path.insert(0, '.')

from freshprice_env.freshprice_env import FreshPriceEnv
from freshprice_env.enums import CurriculumScenario

# Quick smoke test
env = FreshPriceEnv(scenario=CurriculumScenario.STABLE_WEEK, seed=42)
obs, info = env.reset()
print('Environment OK')
print(f'Observation length: {len(obs)} chars')
print(f'Scenario: {info["scenario"]}')

In [ ]:
# Run baselines first (no GPU needed, ~2 min)
!python eval/run_quick_eval.py --agent rule_based --episodes 2
# This gives you the BEFORE numbers for your demo

## 3. Load Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='Qwen/Qwen2.5-7B-Instruct',
    max_seq_length=2048,
    dtype=None,           # auto-detect
    load_in_4bit=True,    # QLoRA
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                  # LoRA rank
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('Model loaded with LoRA adapters')
model.print_trainable_parameters()

## 4. SFT Warm-Start

In [ ]:
from training.sft_trainer import SFTTrainer as FreshPriceSFT

sft = FreshPriceSFT(
    model=model,
    tokenizer=tokenizer,
    output_dir='checkpoints/sft_v1',
)

sft.train()
print('SFT complete. Checkpoint saved to checkpoints/sft_v1')

## 5. Evaluate Before GRPO (Baseline LLM)

In [ ]:
!python eval/run_quick_eval.py \
    --agent llm \
    --checkpoint checkpoints/sft_v1 \
    --episodes 3
# This gives you the AFTER SFT numbers

## 6. GRPO Training — Level 0 (STABLE_WEEK)

In [ ]:
import wandb
wandb.init(project='freshprice-openenv', name='grpo_level0')

from training.grpo_trainer import GRPOTrainer
from freshprice_env.enums import CurriculumScenario

grpo = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    scenario=CurriculumScenario.STABLE_WEEK,
    output_dir='checkpoints/grpo_level0',
    n_episodes=20,           # reduce to 10 on T4
    group_size=4,
    learning_rate=1e-5,
)

grpo.train()
print('GRPO Level 0 complete')

## 7. Evaluate After GRPO

In [ ]:
!python eval/run_quick_eval.py \
    --agent llm \
    --checkpoint checkpoints/grpo_level0 \
    --episodes 5
# This shows WRR improvement from GRPO

## 8. Plot Reward Curves

In [ ]:
import json
import matplotlib.pyplot as plt
import os

os.makedirs('eval/plots', exist_ok=True)

# Load all eval results
results = {}
for fname, label in [
    ('eval/quick_eval_results.json', 'After GRPO'),
]:
    if os.path.exists(fname):
        with open(fname) as f:
            results[label] = json.load(f)

# Baseline comparisons
baselines = {
    'Random': 0.05,
    'Rule-Based': 0.34,
    'SFT Only': 0.48,
    'GRPO Level 0': None,  # fill from results
}

if 'After GRPO' in results:
    grpo_results = results['After GRPO'].get('results', [])
    all_wrrs = [r['wrr']['mean'] for r in grpo_results if 'wrr' in r]
    if all_wrrs:
        import statistics
        baselines['GRPO Level 0'] = statistics.mean(all_wrrs)

fig, ax = plt.subplots(figsize=(8, 5))
names = list(baselines.keys())
values = [v if v is not None else 0.0 for v in baselines.values()]
colors = ['#E53935', '#FF8F00', '#FDD835', '#43A047']

bars = ax.bar(names, values, color=colors, alpha=0.85, edgecolor='white', linewidth=1.5)
ax.axhline(0.70, color='green', linestyle='--', linewidth=2, label='Target WRR = 0.70')

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.01, f'{val:.2f}',
            ha='center', va='bottom', fontweight='bold', fontsize=11)

ax.set_ylim(0, 0.85)
ax.set_ylabel('WRR (Weekly Waste Recovery Rate)', fontsize=12)
ax.set_title('FreshPrice AI — Training Progress', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('eval/plots/training_progress.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to eval/plots/training_progress.png')

## 9. Save & Push to HuggingFace

In [ ]:
from huggingface_hub import login
login()  # Enter your HF token

# Save merged 16-bit model (required — NOT model.save_pretrained after 4-bit)
model.save_pretrained_merged(
    'checkpoints/freshprice_grpo_merged',
    tokenizer,
    save_method='merged_16bit',
)

# Push to Hub
model.push_to_hub_merged(
    'YOUR_HF_USERNAME/freshprice-qwen2.5-7b-grpo',  # replace
    tokenizer,
    save_method='merged_16bit',
)
print('Model pushed to HuggingFace Hub!')

## 10. Full Training Pipeline (Optional — Long Run)

Run the full SFT → GRPO → DPO → Curriculum pipeline:

In [ ]:
# Full pipeline (~6-8 hours on A100)
!python training/train.py \
    --base-model Qwen/Qwen2.5-7B-Instruct \
    --output-dir checkpoints \
    --episodes-per-level 30 \
    --dpo-every 10 \
    --wandb-project freshprice-openenv